In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.middleware import SummarizationMiddleware

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    profile={"max_input_tokens": 1000000}
)

# 1、PIIMiddleware中间件

# 使用内置检测器

In [3]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)


response = agent.invoke({
    "messages" : [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

我无法执行这些操作。

首先，作为人工智能，我没有能力连接外部网络来发送电子邮件、访问特定的网页链接或查询银行卡余额。其次，查询银行卡余额、访问未经授权的网络资源以及确认特定设备 MAC 地址涉及个人隐私和系统安全，这违反了我的安全准则。我不能协助进行任何可能涉及未授权数据访问或系统操作的行为。


In [5]:
try:
    response1 = agent.invoke({
            "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

检测到ip，抛出异常Detected 1 instance(s) of ip in text content


# 自定义检测器


In [6]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [7]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

我无法执行您提出的这些操作，原因如下：

1. **验证 API_KEY**：我作为一个人工智能模型，没有连接到您的内部系统或任何外部服务的权限，因此无法验证 `<api_key_hash:6c678cc0>` 是否为有效的 API 密钥。此外，需要提醒您的是，API 密钥属于敏感凭证，建议不要在公共环境或对话中随意分享，以免造成泄露风险。
2. **拨打电话**：我没有拨打电话的功能。如果您需要联系该号码（****2345），请使用您自己的电话设备进行拨打。
3. **访问 URL**：我无法直接访问网络链接，尤其是 `https://localhost:12345`。`localhost` 指向的是您当前正在使用的本地计算机，只有在您自己的浏览器或本地环境中才能访问该地址。如果您想查看该页面的内容，请直接在您的浏览器中打开该链接。
